## General TO DOs:
Most articles don't get locations from NER Pass, so check if it's even better than the LLM Pass or if it's worth the extra time (roughly double):
* Benchmark LLM Pass vs NER (Time and Accuracy). Compare FAC/ORG/LOC given, how good they are, and time taken.
* Check if NER is necessary on LLM Pass.

Improve LLM:
* Slice texts so they fit LLM's token limit of 2048.
* Add better prompt that allows it to not force a boston location
* Make it give a formatted response so NER isn't necesssary.
* Mentions that it is in boston even if it isn't.
* If it says it isn't in Boston, NER still grabs a location. It shouldn't.

Others:
* Populate list of unwanted locations

* ### Still need to Review Topic Modeling


In [1]:
import re
import json
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
tqdm.pandas()

## Llama 7B

In [2]:
from langchain.chains import LLMChain

# llm
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_community.llms import LlamaCpp

# Prompt
from langchain.chains.prompt_selector import ConditionalPromptSelector
from langchain.prompts import PromptTemplate

# Parser
from langchain_core.output_parsers import StrOutputParser

In [3]:
# TODO: Add better prompt that allows it to not force a boston location
# TODO: Make it give a formatted response so NER isn't necesssary.

# Prompt for LLM to do its geolocation task
prompt = PromptTemplate(
    input_variables=["headline", "body"],
    template="""<<SYS>> \n You are an assistant tasked in geo-locating \
this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
of where you think this article is talking about. BE SPECIFIC AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. Give your response in the following format: \
1.Y/N indicating whether the article is talking about a region of Boston. \n 2.The specific location within the city you got if you got Y in the first question. \
3. The involved specific locations or organizations EXPLICITLY FOUND WITHIN THE ARTICLE that influenced your decision. \
If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. \n\n
Headline: \n\n {headline} \n\n Body: \n\n {body} \n\n [/INST]""",
)

# prompt = PromptTemplate(
#     input_variables=["headline"],
#     template="""<<SYS>> \n You are an assistant tasked in geo-locating \
# this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
# of where you think this article is talking about. BE SPECIFIC AND CONCISE AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. PLEASE CONSIDER THE CONTEXT OF THE ARTICLE. Give your response in the following format: \
# 1. A very brief summary of what the article is talking about. \n 2.The specific location you chose based on the context of the article. \
# If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. \n\n
# Headline: \n\n {headline} \n\n [/INST]""",
# )

# prompt = PromptTemplate(
#     input_variables=["headline"],
#     template="""<<SYS>> \n You are an assistant tasked in geo-locating \
# this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
# of where you think this article is talking about. BE SPECIFIC AND CONCISE AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. Give your response in the following format: \
# 1.Y/N indicating whether the article is talking about a region of Boston \n 2.The specific location within the city you got if you got Y in the first question. \
# If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. PLEASE KEEP YOUR ANSWER SHORT. \n\n
# Headline: \n\n {headline} \n\n Body: \n\n {body}  \n\n [/INST]""",
# )

In [4]:
# Call model. Needs to be downloaded
llama_model_path = "./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf"

In [5]:
llm = LlamaCpp(
    model_path=llama_model_path,
    n_gpu_layers=1,
    n_batch=1024,
    n_ctx=2048,
    f16_kv=True,
    callback_manager=CallbackManager([StreamingStdOutCallbackHandler()]),
    verbose=True,
)
output_parser = StrOutputParser()

llama_model_loader: loaded meta data with 19 key-value pairs and 291 tensors from ./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = LLaMA v2
llama_model_loader: - kv   2:                       llama.context_length u32              = 4096
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 11008
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.head_count u

In [6]:
chain = prompt | llm | output_parser

In [7]:
# Run LLM on a given article
def run_llm(headline, body):
    return chain.invoke({"headline": headline, "body": body})

## NER Model

In [8]:
import spacy
from span_marker import SpanMarkerModel

In [9]:
# Load the spacy model with the span_marker pipeline component
nlp = spacy.load("en_core_web_sm", exclude=["ner"])
nlp.add_pipe("span_marker", config={"model": "tomaarsen/span-marker-roberta-large-ontonotes5"})

## Google Maps

In [10]:
# Load environment variables
import os
from dotenv import load_dotenv

load_dotenv()
gmap_api_key =  os.getenv('GMAP_API_KEY')

In [11]:
import ast
import requests
import googlemaps
from mapbox import Geocoder

In [12]:
gmap_client_key = gmap_api_key
gmaps = googlemaps.Client(key=gmap_client_key)

In [13]:
# Google Maps API handler
def callGoogleMapsAPI(location):
    try:
        # If it's in boston, we can do a more specific search
        if ("Boston" in location):
            location = f"{location}, Boston"
            
        # Locations are limited to Massachusetts for now
        geocode_result = gmaps.geocode(f"{location}, Massachussetts", components={"administrative_area_level": "MA", "country": "US"})
        
        if (len(geocode_result) > 0):
            longitude = geocode_result[0]['geometry']['location']['lng']
            latitude = geocode_result[0]['geometry']['location']['lat']
            return longitude, latitude
        else:
            return None
    except Exception as error:
        print(error)
        return None

### Functions to manage caches

In [14]:
# Load the cache from the file at the start
def load_cache(path):
    try:
        with open(path, 'r') as file:
            cache = json.load(file)
    except FileNotFoundError:
        cache = {}
    return cache

# Save cache to file
def save_cache_to_file(cache, path):
    with open(path, 'w') as file:
        json.dump(cache, file, indent=4)

### Wrapper function to measure time taken by a given function

In [15]:
import time

def check_time(func):
    def sec_to_hms(seconds):
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        remaining_seconds = round(seconds % 60)
        return f"{hours:02}:{minutes:02}:{remaining_seconds:02}"
    
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        total_time = end_time - start_time
        total_time_formatted = sec_to_hms(total_time)
        print(f"Time taken: {total_time_formatted}")
        return result
    return wrapper

## Pipeline Entry Point

In [16]:
sample_data_path = "./sample_data/Articles_Nov_2020_March_2023.csv" # Using this as I don't have the other one
# sample_data_dir = "./sample_data/se_naacp_db.articles_data.csv"

In [17]:
# Temporary. Use given article data set. Comment out when obtain the other data sate
full_df = pd.read_csv(sample_data_path)

# Format data set to match expected pipeline input
full_df = full_df.rename(columns={"Headline": "hl1", "Body": "body"})

# Make 'tagging' column be the id column
tagging_col = full_df.pop('Tagging')
full_df.insert(0, '_id', tagging_col)

# Drop rows where at least one of the specified columns is empty
columns_to_check = ['_id', 'hl1', 'body'] 
full_df = full_df.dropna(subset=columns_to_check, how='all')

# Drop empty rows too
full_df = full_df[~full_df['body'].apply(lambda x: isinstance(x, float))]
full_df = full_df[~full_df['hl1'].apply(lambda x: isinstance(x, float))]

# Pick a random sample of 5 articles
raw_df = full_df.sample(5)
# raw_df = full_df
len(raw_df)


5

In [18]:
# raw_df = pd.read_csv(sample_data_path)

In [19]:
raw_df.head(10)

,_id,Type,Label,hl1,Byline,Section Navigation,Section,Title,Paths,Publish Date,Has Path?,body
1056,00000176-762c-d9b5-af76-766db3a80001,Article,"Friday, December 18","Friday, December 18",00000176-762c-d9b5-af76-766db3a80000,NaN,Digital Mural,NaN,/digital-mural/2020/12/18/friday-december-19 (...,Fri Dec 18 09:16:02 EST 2020,TRUE,"Tonight, In collaboration with the Ella Fitzge..."
4765,0000017a-a776-dc5c-a77a-aff6960b0001,Article,The 'Best Meteor Shower Of The Year' Is Happen...,The 'Best Meteor Shower Of The Year' Is Happen...,Josie Fischels,NaN,Science and Technology,NaN,/science-and-technology/2021/07/15/the-best-me...,Wed Jul 14 19:03:00 EDT 2021,TRUE,"The Perseid Meteor Shower is upon us, and will..."
4779,0000017a-aae9-dd7d-a57b-afe98d680001,Article,The Arroyos — Father And Son — Endorse Kim Jan...,The Arroyos — Father And Son — Endorse Kim Jan...,Saraya Wintersmith,NaN,Politics,NaN,/politics/2021/07/15/the-arroyos-father-and-so...,Thu Jul 15 12:34:16 EDT 2021,TRUE,Boston City Councilor Ricardo Arroyo and his f...
5205,0000017b-3c7c-daa6-ad7f-fdfd1f970001,Article,It 'Felt Like' Nearly 105 Degrees. The Patriot...,It 'Felt Like' Nearly 105 Degrees. The Patriot...,"Arun Rath, Marilyn Schairer",NaN,National News,NaN,/national-news/2021/08/12/it-felt-like-nearly-...,Thu Aug 12 21:30:24 EDT 2021,TRUE,The Red Sox faced off against the Rays on Thur...
3107,00000178-825c-dbb7-a7ff-8a7e85a10001,Article,Prosecutors Add Sex Trafficking Charges Agains...,Prosecutors Add Sex Trafficking Charges Agains...,Jaclyn Diaz,NaN,National News,NaN,/national-news/2021/03/30/prosecutors-add-sex-...,Tue Mar 30 04:51:00 EDT 2021,TRUE,Federal prosecutors in New York filed new char...


The ML Model honestly just needs the `id`, `header`, and `body`.

In [20]:
df = pd.concat([raw_df['_id'], raw_df['hl1'], raw_df['body']], axis=1)

In [21]:
# For Testing Purposes Only
# df = df[:20]

Remove Duplicates (if any)

In [22]:
duplicates = df.duplicated(subset=['hl1'])

In [23]:
print(duplicates.value_counts())

False    5
Name: count, dtype: int64


In [24]:
df = df.drop_duplicates(subset=['hl1'])

Clean the HTML in the body and header

In [25]:
# Function to extract the text from the html of the article
func_clean_html = lambda text: BeautifulSoup(text, "html.parser").get_text()
df['body'] = df['body'].progress_apply(func_clean_html)
df['hl1'] = df['hl1'].progress_apply(func_clean_html)

  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:00<?, ?it/s]


Clean the Body and Header with Regex

In [26]:
# Function to remove extra symbols from the text
func_clean_regex = lambda text: ' '.join([word for word in re.findall(r'[A-Za-z0-9!@#$%^&*().]+', text) if len(word) > 1])
df['body'] = df['body'].progress_apply(func_clean_regex)
df['hl1'] = df['hl1'].progress_apply(func_clean_regex)

100%|██████████| 5/5 [00:00<?, ?it/s]


### Explicit Article Mentions

Load the well-known locations, organizations, and neighborhoods dictionary and the locations that we don't want to allow (i.e. Too broad or incorrect ones like "Boston", "Massachussets", etc.)

In [27]:
known_title_locs_path = "./geodata/known_locs.json"
known_title_locs = load_cache(known_title_locs_path)

unwanted_entities_path = "./geodata/unwanted_locations.json"  
unwanted_entities = load_cache(unwanted_entities_path)

In [28]:
# If a location is in the title, use that as the article's location
def explicit_filtering(header):
    # Look through the header for known locations
    lowercase_header = header.lower()
    for location in known_title_locs.keys():
        if (location.lower() in lowercase_header):
            if location not in unwanted_entities["FAC"]:
                return location
    return None

In [29]:
df["Explicit_Pass"] = df["hl1"].progress_apply(explicit_filtering)

100%|██████████| 5/5 [00:00<?, ?it/s]


In [30]:
df["Explicit_Pass"].value_counts().head(10)

Explicit_Pass
BOSTON    1
Name: count, dtype: int64

### NER Code First Pass

In [31]:
# Return the first valid facility found, or organization if none are found
def valid_facility(entities, firstPass):
    print(entities)

    if (firstPass): 
        for entity in entities:
            # If it's a valid facility, return it
            if (entity.label_ == "FAC" and entity.text not in unwanted_entities["FAC"]):
                return entity.text
        else:
            return None
    
    # Process for the LLM Prediction Pass
    else:
        first_org = None
        for entity in entities:
            # If it's a valid facility, return it
            if (entity.label_ == "FAC" and entity.text not in unwanted_entities["FAC"]):
                return entity.text
            
            # If it's a valid organization, save it (but don't return in case there's a facility later on)
            if (first_org == None and entity.label_ == "ORG" and entity.text not in unwanted_entities["ORG"]):
                first_org = entity.text
        else:             
            return first_org # Return regardless of whether it's None or not 
        

In [32]:
# Run NER on the body of the article and return first valid facility
def run_NER(text, firstPass=True):
    try:
        if (text == None or text == ""):
            return None
        
        entities = nlp(text).ents
        return valid_facility(entities, firstPass)
        
    except Exception as error:
        print(error)
        return None

In [33]:
# Old way of running the NER
# Run NER on the articles that do not have an explicit location in the title
def explicit_filtering_NER(article):
    try:
        # If the article does not have an explicit location, run NER
        if (article['Explicit_Pass'] != None): 
            print(f"Has location from title: {article['hl1']}")
            return None
        else:
            return run_NER(article['body'])
    except Exception as error:
        print(error)
        return None

# df['NER_Pass'] = df.progress_apply(explicit_filtering_NER, axis=1)

In [34]:
# Chunk processing - split the article into chunks of text and run NER on each chunk
chunk_size = 100
def chunk_processing(text, chunk_size=chunk_size):
    # Split text into smaller chunks
    chunks = split_text_into_chunks(text, chunk_size)

    # Process each chunk and return if a valid facilty is found
    for chunk in chunks:
        result = run_NER(chunk)
        if result is not None:
             return result
    return None

# Split article text into chunks of specified size
def split_text_into_chunks(text, chunk_size=chunk_size):
    words = text.split()
    chunks = [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]
    return chunks

In [35]:
@check_time
def handle_chunk_processing(article):
    # Filter out articles that have an explicit location in the title
    if (article['Explicit_Pass'] != None): 
        print(f"Has location from title: {article['hl1']}")
        return None

    # Filter out articles that have no text
    text = article['body']
    if (text is None or text == ""):
            return None
    try:
        return chunk_processing(text)
    except Exception as error:
        print(error)

In [36]:
df["NER_Pass"] = df.progress_apply(handle_chunk_processing, axis=1)

  0%|          | 0/5 [00:00<?, ?it/s]

(Tonight, the Ella Fitzgerald Foundation, the American Pops Orchestra, Ella Fitzgerald, 1960, Ella Wishes You Swinging Christmas, One, American, Vanessa Williams, Dee Dee Bridgewater, Norm Lewis, Carmen Ruby Floyd, today, Nova Payton, Dave Detwiler, Morgan James, Ella, Christmas, 10pm, Kevin Parisi, American Pops)


 40%|████      | 2/5 [00:19<00:28,  9.57s/it]

(Orchestra Inc,)
Time taken: 00:00:19
(The Perseid Meteor Shower, August 24, the year, NASA, August 12, up to 100, 37 miles)
(The Perseid shower, Earth, 109PSwift Tuttle comet, 2125, Perseus)
(NASA, the night of August 11, August 12, the pre dawn hours, Northern Hemisphere, NASA, NASA, International Dark Sky Association)


 60%|██████    | 3/5 [01:21<01:03, 31.61s/it]

(as early as 10 p.m., any night, Josie Fischels, NPR News Desk, 2021, NPR)
Time taken: 00:01:02
Has location from title: The Arroyos Father And Son Endorse Kim Janey For Boston Mayor
Time taken: 00:00:00


100%|██████████| 5/5 [01:36<00:00, 18.54s/it]

(The Red Sox, Rays, Thursday, afternoon, Fenway, Patriots, the Washington Football Team, Gillette Stadium, nearly 105 degrees, Massachusetts, GBH All Things Considered, Arun Rath, Joseph Van Allen, the Boston Sports)
Time taken: 00:00:15
(New York, Monday, Ghislaine Maxwell, Jeffrey Epstein, Maxwell, 14 year old, Epstein, from 2001 to 2004, Epstein, Palm Beach, Florida, 14 year old, Epstein, Maxwell, Epstein)
(Epstein, two, Maxwell, Epstein, Maxwell, the 1990s, 2001, 2004, Maxwell)
(Maxwell, last July, New Hampshire, nearly year, Epstein, About month, Epstein, Manhattan, Maxwell, Epstein, Maxwell)
(Epstein, Maxwell, Florida, Epstein, hundreds of dollars, Epstein, Maxwell, New York, Epstein, Maxwell, Epstein, under 18)


100%|██████████| 5/5 [02:54<00:00, 34.93s/it]

(Epstein, Palm Beach, hundreds of dollars, Epstein, Maxwell, 2021, NPR)
Time taken: 00:01:18


In [37]:
df.head(10)

,_id,hl1,body,Explicit_Pass,NER_Pass
1056,00000176-762c-d9b5-af76-766db3a80001,Friday December 18,Tonight In collaboration with the Ella Fitzger...,None,None
4765,0000017a-a776-dc5c-a77a-aff6960b0001,The Best Meteor Shower Of The Year Is Happenin...,The Perseid Meteor Shower is upon us and will ...,None,None
4779,0000017a-aae9-dd7d-a57b-afe98d680001,The Arroyos Father And Son Endorse Kim Janey F...,Boston City Councilor Ricardo Arroyo and his f...,BOSTON,None
5205,0000017b-3c7c-daa6-ad7f-fdfd1f970001,It Felt Like Nearly 105 Degrees. The Patriots ...,The Red Sox faced off against the Rays on Thur...,None,Fenway
3107,00000178-825c-dbb7-a7ff-8a7e85a10001,Prosecutors Add Sex Trafficking Charges Agains...,Federal prosecutors in New York filed new char...,None,None


### Llama Prediction

In [38]:
#TODO: Comply with token limit of 2048 for Llama
# Run the LLM model on the articles that haven't been tagged with a location yet. Then run NER on the LLM prediction
@check_time
def predict_llama(article):
    try:
        # If the article does not have an explicit location or NER location, run LLM
        if (article['Explicit_Pass'] != None or article['NER_Pass'] != None):
            print(f"Has location from title or NER: {article['hl1']}")
            return None
        else:
            llama_prediction = run_llm(article['hl1'], article['body'])
            print(llama_prediction)
            return run_NER(llama_prediction, False)
    except Exception as error:
        print(error)
        return None

In [39]:
df['NER_Prediction'] = df.progress_apply(predict_llama, axis=1)

  0%|          | 0/5 [00:00<?, ?it/s]

  Response:

1. Y - The article is talking about a specific region of Boston, specifically the area of Dorchester.
2. The specific location within Dorchester is the historic Middle East nightclub, which is mentioned in the article as the venue for the concert.
3. The involved specific locations or organizations explicitly found within the article that influenced my decision are:
* The Ella Fitzgerald Foundation
* The American Pops Orchestra
* GBH 2 (the television station airing the concert)

Based on these details, I believe the article is likely referring to a concert held at the Middle East nightclub in Dorchester, Boston, featuring a star-studded lineup of artists performing Ella Fitzgerald's classic holiday album "Ella Wishes You Swinging Christmas".


llama_print_timings:        load time =   40021.10 ms
llama_print_timings:      sample time =      75.52 ms /   174 runs   (    0.43 ms per token,  2304.06 tokens per second)
llama_print_timings: prompt eval time =   40018.23 ms /   382 tokens (  104.76 ms per token,     9.55 tokens per second)
llama_print_timings:        eval time =   33921.11 ms /   173 runs   (  196.08 ms per token,     5.10 tokens per second)
llama_print_timings:       total time =   74728.80 ms /   555 tokens


  Response:

1. Y - The article is talking about a specific region of Boston, specifically the area of Dorchester.
2. The specific location within Dorchester is the historic Middle East nightclub, which is mentioned in the article as the venue for the concert.
3. The involved specific locations or organizations explicitly found within the article that influenced my decision are:
* The Ella Fitzgerald Foundation
* The American Pops Orchestra
* GBH 2 (the television station airing the concert)

Based on these details, I believe the article is likely referring to a concert held at the Middle East nightclub in Dorchester, Boston, featuring a star-studded lineup of artists performing Ella Fitzgerald's classic holiday album "Ella Wishes You Swinging Christmas".


 40%|████      | 2/5 [01:55<02:53, 57.67s/it]

...

  1. Y - The article is talking about a region of New York City, specifically the Palm Beach estate in Florida.
2. The specific location within the city is the Palm Beach estate owned by Jeffrey Epstein, located in Palm Beach, Florida.
3. The involved specific locations or organizations explicitly found within the article that influenced my decision are:
* Palm Beach estate in Florida, where the alleged abuse took place
* New York City, where the federal prosecutors filed the new charges against Ghislaine Maxwell
* Other locations where Epstein and Maxwell encouraged the 14-year-old girl to bring her friends or recruit other young women to provide massages to Epstein.


llama_print_timings:        load time =   40021.10 ms
llama_print_timings:      sample time =      56.25 ms /   157 runs   (    0.36 ms per token,  2791.21 tokens per second)
llama_print_timings: prompt eval time =   56938.18 ms /   660 tokens (   86.27 ms per token,    11.59 tokens per second)
llama_print_timings:        eval time =   29163.85 ms /   156 runs   (  186.95 ms per token,     5.35 tokens per second)
llama_print_timings:       total time =   86528.39 ms /   816 tokens


  1. Y - The article is talking about a region of New York City, specifically the Palm Beach estate in Florida.
2. The specific location within the city is the Palm Beach estate owned by Jeffrey Epstein, located in Palm Beach, Florida.
3. The involved specific locations or organizations explicitly found within the article that influenced my decision are:
* Palm Beach estate in Florida, where the alleged abuse took place
* New York City, where the federal prosecutors filed the new charges against Ghislaine Maxwell
* Other locations where Epstein and Maxwell encouraged the 14-year-old girl to bring her friends or recruit other young women to provide massages to Epstein.


100%|██████████| 5/5 [06:04<00:00, 72.98s/it]

(1, New York City, Palm Beach, Florida, 2, Palm Beach, Jeffrey Epstein, Palm Beach, Florida, 3, Palm Beach, Florida, New York City, Ghislaine Maxwell, Epstein, Maxwell, 14-year-old, Epstein)
Time taken: 00:02:00


In [40]:
df.head(10)

,_id,hl1,body,Explicit_Pass,NER_Pass,NER_Prediction
1056,00000176-762c-d9b5-af76-766db3a80001,Friday December 18,Tonight In collaboration with the Ella Fitzger...,None,None,Middle East
4765,0000017a-a776-dc5c-a77a-aff6960b0001,The Best Meteor Shower Of The Year Is Happenin...,The Perseid Meteor Shower is upon us and will ...,None,None,the Museum of Fine Arts
4779,0000017a-aae9-dd7d-a57b-afe98d680001,The Arroyos Father And Son Endorse Kim Janey F...,Boston City Councilor Ricardo Arroyo and his f...,BOSTON,None,None
5205,0000017b-3c7c-daa6-ad7f-fdfd1f970001,It Felt Like Nearly 105 Degrees. The Patriots ...,The Red Sox faced off against the Rays on Thur...,None,Fenway,None
3107,00000178-825c-dbb7-a7ff-8a7e85a10001,Prosecutors Add Sex Trafficking Charges Agains...,Federal prosecutors in New York filed new char...,None,None,None


In [41]:
## TODO: DELETE AFTER POPULATING THE UNWANTED ENTITIES CACHE
# unwanted_entities = {
#     'FAC': ['Boston'],
#     'ORG': ['New York Times'],
#     'LOC': ['Boston', 'Massachusetts', 'Brookline', 'Allston'],
#     'GPE': ['Boston', 'Massachusetts', 'Brookline', 'Allston'],
# }

# save_cache_to_file(unwanted_entities, unwanted_entities_path)

Extract locations from the most specific pass

In [42]:
# Get the locations from the most specific pass for a given article
def extractLocations(article):
    for key in ['Explicit_Pass', 'NER_Pass', 'NER_Prediction']:
        location = article.get(key)
        if location is not None:
            return location
    return None

In [43]:
df['Locations'] = df.progress_apply(extractLocations, axis=1)

100%|██████████| 5/5 [00:00<00:00, 1381.43it/s]

In [44]:
df.head(10)

,_id,hl1,body,Explicit_Pass,NER_Pass,NER_Prediction,Locations
1056,00000176-762c-d9b5-af76-766db3a80001,Friday December 18,Tonight In collaboration with the Ella Fitzger...,None,None,Middle East,Middle East
4765,0000017a-a776-dc5c-a77a-aff6960b0001,The Best Meteor Shower Of The Year Is Happenin...,The Perseid Meteor Shower is upon us and will ...,None,None,the Museum of Fine Arts,the Museum of Fine Arts
4779,0000017a-aae9-dd7d-a57b-afe98d680001,The Arroyos Father And Son Endorse Kim Janey F...,Boston City Councilor Ricardo Arroyo and his f...,BOSTON,None,None,BOSTON
5205,0000017b-3c7c-daa6-ad7f-fdfd1f970001,It Felt Like Nearly 105 Degrees. The Patriots ...,The Red Sox faced off against the Rays on Thur...,None,Fenway,None,Fenway
3107,00000178-825c-dbb7-a7ff-8a7e85a10001,Prosecutors Add Sex Trafficking Charges Agains...,Federal prosecutors in New York filed new char...,None,None,None,None


In [45]:
df

,_id,hl1,body,Explicit_Pass,NER_Pass,NER_Prediction,Locations
1056,00000176-762c-d9b5-af76-766db3a80001,Friday December 18,Tonight In collaboration with the Ella Fitzger...,None,None,Middle East,Middle East
4765,0000017a-a776-dc5c-a77a-aff6960b0001,The Best Meteor Shower Of The Year Is Happenin...,The Perseid Meteor Shower is upon us and will ...,None,None,the Museum of Fine Arts,the Museum of Fine Arts
4779,0000017a-aae9-dd7d-a57b-afe98d680001,The Arroyos Father And Son Endorse Kim Janey F...,Boston City Councilor Ricardo Arroyo and his f...,BOSTON,None,None,BOSTON
5205,0000017b-3c7c-daa6-ad7f-fdfd1f970001,It Felt Like Nearly 105 Degrees. The Patriots ...,The Red Sox faced off against the Rays on Thur...,None,Fenway,None,Fenway
3107,00000178-825c-dbb7-a7ff-8a7e85a10001,Prosecutors Add Sex Trafficking Charges Agains...,Federal prosecutors in New York filed new char...,None,None,None,None


## Get the coordinates

In [46]:
known_locations_path = "./geodata/known_locations.json"  
known_locations = load_cache(known_locations_path)

In [47]:
# Get the coordinates of the location
def getCoordinates(location): # Valid labels are FAC for NER_Pass; FAC and ORG for NER_Prediction
    if (location == None or len(location) == 0): return None  
    
    # Only get coordinates if the location is not already known
    if (location in known_locations):
        longitude, latitude = known_locations[location]["coordinates"]
    else:
        # Get coordinates and save to cache
        longitude, latitude = callGoogleMapsAPI(location)
        known_locations[location] = {"coordinates": [longitude, latitude], "tract": None, "county": None}
        save_cache_to_file(known_locations, known_locations_path)

    return [longitude, latitude]

In [48]:
df['Coordinates'] = df['Locations'].progress_apply(getCoordinates)

  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:00<00:00,  6.46it/s]


In [49]:
df.head(10)

,_id,hl1,body,Explicit_Pass,NER_Pass,NER_Prediction,Locations,Coordinates
1056,00000176-762c-d9b5-af76-766db3a80001,Friday December 18,Tonight In collaboration with the Ella Fitzger...,None,None,Middle East,Middle East,"[42.5509603, 29.2985278]"
4765,0000017a-a776-dc5c-a77a-aff6960b0001,The Best Meteor Shower Of The Year Is Happenin...,The Perseid Meteor Shower is upon us and will ...,None,None,the Museum of Fine Arts,the Museum of Fine Arts,"[-71.094048, 42.339381]"
4779,0000017a-aae9-dd7d-a57b-afe98d680001,The Arroyos Father And Son Endorse Kim Janey F...,Boston City Councilor Ricardo Arroyo and his f...,BOSTON,None,None,BOSTON,"[-71.0588801, 42.3600825]"
5205,0000017b-3c7c-daa6-ad7f-fdfd1f970001,It Felt Like Nearly 105 Degrees. The Patriots ...,The Red Sox faced off against the Rays on Thur...,None,Fenway,None,Fenway,"[-71.0923139, 42.346832]"
3107,00000178-825c-dbb7-a7ff-8a7e85a10001,Prosecutors Add Sex Trafficking Charges Agains...,Federal prosecutors in New York filed new char...,None,None,None,None,None


## Geocode locations

In [50]:
# Get the census tract of the location
def query_census_api(location, coordinates):
    longitude, latitude = coordinates
    base_url = f'https://geocoding.geo.census.gov/geocoder/geographies/coordinates?'
    survey_ver = f'&benchmark=4&vintage=4&layers=2020 Census Blocks&format=json'
    url = f'{base_url}x={longitude}&y={latitude}{survey_ver}'

    response = requests.get(url)

    # Check if response is valid
    if (response.status_code == 200):
        results = response.json()
        try:
            tract = results['result']['geographies']['2020 Census Blocks'][0]['TRACT']
            county = results['result']['geographies']['2020 Census Blocks'][0]['COUNTY']

            return tract, county
        except IndexError:
            print("Unable to retrieve census geography for: " + location)
        except KeyError:
            print("Location is outside of the United States: " + location)
        except Exception as error:
            print(error)

    return None, None  # Return this if API call failed or no tracts found

In [51]:
# Get the census tract and county of the location
def geocode(location):
    if (location is None or len(location) == 0): return None, None  

    # Only geocode if it's not known
    Tract = known_locations[location]["tract"]
    County = known_locations[location]["county"]
    if (Tract is None or County is None):
        # Geocode article
        coordinates = known_locations[location]["coordinates"]
        Tract, County = query_census_api(location, coordinates)

        # Save to cache
        known_locations[location]["tract"] = Tract
        known_locations[location]["county"] = County
        save_cache_to_file(known_locations, known_locations_path)
    
    return Tract, County
    

In [52]:
df[['Tracts', 'County']] = df.progress_apply(lambda row: pd.Series(geocode(row['Locations'])), axis=1)
df

 40%|████      | 2/5 [00:00<00:00,  3.91it/s]

Location is outside of the United States: Middle East


100%|██████████| 5/5 [00:01<00:00,  3.94it/s]


,_id,hl1,body,Explicit_Pass,NER_Pass,NER_Prediction,Locations,Coordinates,Tracts,County
1056,00000176-762c-d9b5-af76-766db3a80001,Friday December 18,Tonight In collaboration with the Ella Fitzger...,None,None,Middle East,Middle East,"[42.5509603, 29.2985278]",None,None
4765,0000017a-a776-dc5c-a77a-aff6960b0001,The Best Meteor Shower Of The Year Is Happenin...,The Perseid Meteor Shower is upon us and will ...,None,None,the Museum of Fine Arts,the Museum of Fine Arts,"[-71.094048, 42.339381]",010300,025
4779,0000017a-aae9-dd7d-a57b-afe98d680001,The Arroyos Father And Son Endorse Kim Janey F...,Boston City Councilor Ricardo Arroyo and his f...,BOSTON,None,None,BOSTON,"[-71.0588801, 42.3600825]",030302,025
5205,0000017b-3c7c-daa6-ad7f-fdfd1f970001,It Felt Like Nearly 105 Degrees. The Patriots ...,The Red Sox faced off against the Rays on Thur...,None,Fenway,None,Fenway,"[-71.0923139, 42.346832]",981800,025
3107,00000178-825c-dbb7-a7ff-8a7e85a10001,Prosecutors Add Sex Trafficking Charges Agains...,Federal prosecutors in New York filed new char...,None,None,None,None,None,None,None


In [53]:
print(df['Explicit_Pass'].value_counts().sum())
df['Explicit_Pass'].value_counts()

1


Explicit_Pass
BOSTON    1
Name: count, dtype: int64

In [54]:
print(df['NER_Pass'].value_counts().sum())
df['NER_Pass'].value_counts()

1


NER_Pass
Fenway    1
Name: count, dtype: int64

In [55]:
print(df['NER_Prediction'].value_counts().sum())
df['NER_Prediction'].value_counts()

2


NER_Prediction
Middle East                1
the Museum of Fine Arts    1
Name: count, dtype: int64

In [56]:
df.head(10)

,_id,hl1,body,Explicit_Pass,NER_Pass,NER_Prediction,Locations,Coordinates,Tracts,County
1056,00000176-762c-d9b5-af76-766db3a80001,Friday December 18,Tonight In collaboration with the Ella Fitzger...,None,None,Middle East,Middle East,"[42.5509603, 29.2985278]",None,None
4765,0000017a-a776-dc5c-a77a-aff6960b0001,The Best Meteor Shower Of The Year Is Happenin...,The Perseid Meteor Shower is upon us and will ...,None,None,the Museum of Fine Arts,the Museum of Fine Arts,"[-71.094048, 42.339381]",010300,025
4779,0000017a-aae9-dd7d-a57b-afe98d680001,The Arroyos Father And Son Endorse Kim Janey F...,Boston City Councilor Ricardo Arroyo and his f...,BOSTON,None,None,BOSTON,"[-71.0588801, 42.3600825]",030302,025
5205,0000017b-3c7c-daa6-ad7f-fdfd1f970001,It Felt Like Nearly 105 Degrees. The Patriots ...,The Red Sox faced off against the Rays on Thur...,None,Fenway,None,Fenway,"[-71.0923139, 42.346832]",981800,025
3107,00000178-825c-dbb7-a7ff-8a7e85a10001,Prosecutors Add Sex Trafficking Charges Agains...,Federal prosecutors in New York filed new char...,None,None,None,None,None,None,None


In [57]:
len(df)

5

In [58]:
df = df.dropna(subset=["Tracts", "County"]) # Clean those that don't have a Tract or a County

In [59]:
print(len(df))
df.head(10)

3


,_id,hl1,body,Explicit_Pass,NER_Pass,NER_Prediction,Locations,Coordinates,Tracts,County
4765,0000017a-a776-dc5c-a77a-aff6960b0001,The Best Meteor Shower Of The Year Is Happenin...,The Perseid Meteor Shower is upon us and will ...,None,None,the Museum of Fine Arts,the Museum of Fine Arts,"[-71.094048, 42.339381]",010300,025
4779,0000017a-aae9-dd7d-a57b-afe98d680001,The Arroyos Father And Son Endorse Kim Janey F...,Boston City Councilor Ricardo Arroyo and his f...,BOSTON,None,None,BOSTON,"[-71.0588801, 42.3600825]",030302,025
5205,0000017b-3c7c-daa6-ad7f-fdfd1f970001,It Felt Like Nearly 105 Degrees. The Patriots ...,The Red Sox faced off against the Rays on Thur...,None,Fenway,None,Fenway,"[-71.0923139, 42.346832]",981800,025


In [ ]:
df.to_csv("./sample_data/geocoded_output.csv")

## Topic Modeling

In [60]:
import os
import tiktoken
import numpy as np
from transformers import pipeline
from sklearn.metrics import adjusted_rand_score
from openai import OpenAI, AsyncOpenAI
from sklearn.metrics.pairwise import cosine_similarity
from tenacity import retry, wait_random_exponential, stop_after_attempt

## OpenAI Client

In [61]:
# Retry up to 10 times with exponential backoff, starting at 1 second and maxing out at 20 seconds delay
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(10))
def get_embedding(text: str, model="text-embedding-3-small"):
    #print(text)
    try:
        embedding = client.embeddings.create(input=text, model=model).data[0].embedding
        return embedding
    except Exception as e:
        print(f"Failed to retrieve ADA Embedding: {e}. Replacing with replacement value!")
        return [-1.0]
    return 

In [62]:
client = OpenAI(
    api_key='YOUR_KEY_HERE',
)

## Taxonomy Lists

Content Taxanomy

In [63]:
# Get the embedding for taxonomy
taxonomy_df = pd.read_csv('./taxonomy_list/Content_Taxonomy.csv', skiprows=5, usecols=range(8))
taxonomy_df.columns = taxonomy_df.iloc[0]
taxonomy_df = taxonomy_df.tail(-1)

tier_1_list = []
tier_2_list = []
tier_3_list = []
tier_4_list = []
for index, row in taxonomy_df.iterrows():
    if not pd.isnull(row['Tier 4']) and row['Tier 4'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_3_label = row['Tier 3']
        tier_4_label = row['Tier 4']
        tier_4_list.append(f'{tier_1_label} - {tier_2_label} - {tier_3_label} - {tier_4_label}')
    elif not pd.isnull(row['Tier 3']) and row['Tier 3'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_3_label = row['Tier 3']
        tier_3_list.append(f'{tier_1_label} - {tier_2_label} - {tier_3_label}')
    elif not pd.isnull(row['Tier 2']) and row['Tier 2'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_2_list.append(f'{tier_1_label} - {tier_2_label}')
    else:
        tier_1_label = row['Tier 1']
        tier_1_list.append(f'{tier_1_label}')

tier_1_list = list(set(tier_1_list))
tier_2_list = list(set(tier_2_list))
tier_3_list = list(set(tier_3_list))
tier_4_list = list(set(tier_4_list))

tier_1_embedding = [get_embedding(topic) for topic in tier_1_list]
tier_2_embedding = [get_embedding(topic) for topic in tier_2_list]
tier_3_embedding = [get_embedding(topic) for topic in tier_3_list]
tier_4_embedding = [get_embedding(topic) for topic in tier_4_list]

all_topics_list = []
[all_topics_list.append(topic) for topic in tier_1_list]
[all_topics_list.append(topic) for topic in tier_2_list]
[all_topics_list.append(topic) for topic in tier_3_list]
[all_topics_list.append(topic) for topic in tier_4_list]

all_topics_embedding = []
[all_topics_embedding.append(embedding) for embedding in tier_1_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_2_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_3_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_4_embedding]
print(len(all_topics_embedding))

FileNotFoundError: [Errno 2] No such file or directory: './taxonomy_list/Content_Taxonomy.csv'

Selected Taxonomy List

In [ ]:
# Get embedding for the 230 topics selected by BERTopic 
selected_taxonomy_df = pd.read_csv('./topics/embedding_similarity_label.csv')
selected_taxonomy_df = selected_taxonomy_df.dropna(subset=['closest_topic'])
selected_topics_list = selected_taxonomy_df['closest_topic'].values.tolist()

selected_topics_embedding = [get_embedding(topic) for topic in selected_topics_list]

Client Taxonomy List

In [ ]:
# Alternative taxonomy: client's list of topics
client_taxonomy_df = pd.read_excel('./topics/Asad_Topics_List.xlsx', names=['label'])
client_taxonomy_df['ada_embedding'] = client_taxonomy_df['label'].map(get_embedding)

## Obtaining Ada Embedding

In [ ]:
def truncate(tokens, length=500):
    """
    Function to get the first 500 elements from a list
    """
    return tokens[:length]

In [ ]:
df['topic_model_body'] = df['body'].apply(lambda x: re.sub(re.compile('<.*?>'), '', x))
df['tokens'] = df['topic_model_body'].apply(lambda x: x.split())
df['tokens'] = df['tokens'].apply(truncate)

In [ ]:
df['ada_embedding'] = df.tokens.apply(lambda x: get_embedding(','.join(map(str,x)), model='text-embedding-3-small'))

## Similarity Matching After Ada Embedding

In [ ]:
# Find most similar taxonomy (out of all toipcs) to news body
closest_topic_list_all = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in all_topics_embedding]

    # Find the index of the topic with the highest similarity
    closest_topic_index = np.argmax(similarities)

    # Retrieve the closest topic embedding
    closest_topic = all_topics_list[closest_topic_index]
    closest_topic_list_all.append(closest_topic)

df['closest_topic_all'] = closest_topic_list_all

In [ ]:
# Find most similar taxonomy (out of 230 selected topics) to news body
closest_topic_list_selected = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in selected_topics_embedding]

    # Find the index of the topic with the highest similarity
    closest_topic_index = np.argmax(similarities)

    # Retrieve the closest topic embedding
    closest_topic = selected_topics_list[closest_topic_index]
    closest_topic_list_selected.append(closest_topic)

df['closest_topic_selected'] = closest_topic_list_selected

In [ ]:
client_topic_embedding_list = client_taxonomy_df['ada_embedding'].to_list()
client_topic_list = client_taxonomy_df['label'].to_list()
similarity_arr = []

closest_topic_list_client = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in client_topic_embedding_list]
    
    if max(similarities) > 0.25:    
        closest_topic_index = np.argmax(similarities) # Find the index of the topic with the highest similarity
        closest_topic = client_topic_list[closest_topic_index] # Retrieve the closest topic embedding
        closest_topic_list_client.append(closest_topic)
    else:
        closest_topic_list_client.append('Other')
    similarity_arr.append(max(similarities))
    
df['closest_topic_client'] = closest_topic_list_client

In [ ]:
df

In [ ]:
df.to_csv("./outputs/gbh_output.csv")

In [ ]:
raw_df

In [ ]:
df

In [ ]:
merged_df = pd.merge(raw_df, df, on='_id', how='inner')

In [ ]:
merged_df

In [ ]:
merged_df.to_csv("./outputs/gbh_output_all_fields.csv")

In [ ]:
merged_df.columns